Microservices - Big Ball of Mud (Synchron) - **Open Telemetry Variante**
-------------------------------

![](images/BigBallofMud.png)

- - -

Das Beispiel besteht aus Microservices: **Order**, **Customer**, **Catalog**, **Shipment**, **Invoicing** und **Sales**.

**Order** nutzt **Catalog** und **Customer** mittels deren REST-Schnittstelle. 
**Shipment**, **Invoicing** und **Sales** holen Daten von **Order**, **Catalog** und **Customer** mittels deren REST-Schnittstelle. 

Ausserdem bietet jeder Microservice einige HTML-Seiten an.

Zusätzlich ist ein Service **Webshop** am laufen, der dem Benutzer mit einer Webseite einen einfachen Einstieg in das System ermöglicht.

- - -

Zuerst erstellen wir den Kubernetes Namespace ms-obbm und Labeln diesen, dass Kiali Informationen sammeln kann.

In [ ]:
! kubectl create namespace ms-obbm

Jetzt ist ein guter Zeitpunkt um [HeadLamp](https://headlamp.dev/) (Nachfolger Kubernetes Dashboard) zu starten.

HeadLamp braucht einen Token welcher unter dem URL ausgegeben wird. Diesen markieren und mittels Ctrl+c kopieren und in der Headlamp Loginmaske wieder einfügen -> Ctrl+v.

In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

Wir definieren eine OpenTelemetry-Instrumentation für den Namespace `ms-obbm`, der die Python-Pods automatisch instrumentiert und ihre Traces per OTLP an den Collector sendet. 

Dabei werden Kontext-Propagation (W3C) und Sampling konfiguriert, ohne dass Anpassungen am Anwendungscode nötig sind.D.h. die Services geben Trace-Informationen nach [W3C](https://www.w3.org/TR/trace-context/) weiter, damit Requests über mehrere Systeme verfolgbar sind.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1alpha1
kind: Instrumentation
metadata:
  name: python-auto
  namespace: ms-obbm
spec:
  exporter:
    endpoint: http://otel-collector-collector.opentelemetry.svc.cluster.local:4318

  propagators:
    - tracecontext
    - baggage

  sampler:
    type: parentbased_traceidratio
    argument: "0.5"

  python:
    image: ghcr.io/open-telemetry/opentelemetry-operator/autoinstrumentation-python:0.39b0
    env:
      - name: OTEL_EXPORTER_OTLP_PROTOCOL
        value: http/protobuf
      - name: OTEL_TRACES_EXPORTER
        value: otlp
      - name: OTEL_METRICS_EXPORTER
        value: none
      - name: OTEL_LOGS_EXPORTER
        value: none
      - name: OTEL_RESOURCE_ATTRIBUTES
        value: service.namespace=ms-obbm
EOF

Der Webshop, aus den vorherigen Übungen, ist jetzt aufgeteilt in einzelne Microservices.

Diese starten wir deklarativ mittels `kubectl apply -f <Deklaration>`

Damit OpenTelemetry diese erkennt und automatisch instrumentiert wurde die Annotation `instrumentation.opentelemetry.io/inject-python` und `prometheus.io` hinzugefügt:

      template:
        metadata:
          labels:
            app: catalog
          annotations:
            instrumentation.opentelemetry.io/inject-python: "python-auto"
            prometheus.io/scrape: "true"
            prometheus.io/path: "/metrics"
            prometheus.io/port: "8080"

Für eine volle Unterstützung von OpenTelemetry sind programmtechnische Erweiterungen nötig:
* [Language APIs & SDKs](https://opentelemetry.io/docs/languages/)

Und für Prometheus Metrics Daten:
* [Client libraries](https://prometheus.io/docs/instrumenting/clientlibs/)

**Hinweis**: nach diesen Erweiterungen kann die Annotation `instrumentation.opentelemetry.io/inject-python` entfernt werden.

In [ ]:
%%bash
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/order-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/catalog-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/customer-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/order-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/webshop-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/shipment-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/invoicing-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/sales-deployment.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/shipment-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/invoicing-service.yaml
kubectl apply --namespace ms-obbm -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/sales-service.yaml
kubectl get pod,services --namespace ms-obbm

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-obbm webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

---
## OpenTelemetry Observability Stack (Jaeger mit Monitoring-Stack: Prometheus, Grafana, AlertManager)

![](https://opentelemetry.io/img/otel-diagram.svg)

Quelle: [Opentelemetry](https://opentelemetry.io/docs/)

---

Traces laufen über den OpenTelemetry Collector nach Jaeger.

Opentelemetry scrappt die Metriken von den Microservices und leitet sie an Prometheus weiter.

Grafana mit seinen vorkonfigurierten Dashboard liest die Daten aus Prometheus und Jaeger und zeigt diese an.

In [ ]:
%%bash
NAMESPACE="opentelemetry"
SERVER_IP="$(cat ~/data/server-ip 2>/dev/null || true)"
JAEGER_PORT="$(kubectl -n "${NAMESPACE}" get svc jaeger -o=jsonpath='{.spec.ports[?(@.port==16686)].nodePort}')"
PROMETHEUS_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-prometheus -o=jsonpath='{.spec.ports[?(@.port==9090)].nodePort}')"
ALERTMANAGER_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-alertmanager -o=jsonpath='{.spec.ports[?(@.port==9093)].nodePort}')"
GRAFANA_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-grafana -o=jsonpath='{.spec.ports[?(@.port==80)].nodePort}')"

echo "Jaeger UI       : http://${SERVER_IP}:${JAEGER_PORT}"
echo "Prometheus UI   : http://${SERVER_IP}:${PROMETHEUS_PORT}"
echo "Alertmanager UI : http://${SERVER_IP}:${ALERTMANAGER_PORT}"
echo "Grafana UI      : http://${SERVER_IP}:${GRAFANA_PORT}"

- - -
## Lasttest

Um die Verbindungen sichtbar zu machen, erzeugen wir ein wenig Traffic.

In [ ]:
%%bash
URL="http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-obbm webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

for i in `seq 1 60`;
do
    ab -t 1s -c 10  ${URL}/order/order
    ab -t 1s -c 10  ${URL}/order/order
    ab -t 1s -c 10  ${URL}/shipment/shipment
    ab -t 1s -c 10  ${URL}/invoicing/invoicing
    ab -t 1s -c 10  ${URL}/sales/sales
    echo
done

- - -

Aufräumen

In [ ]:
! kubectl delete namespace ms-obbm

- - -
### Quellen

* Sourcecode: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/-/tree/v3.2.0?ref_type=heads
* Kubernetes Deklarationen: https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates
* Container Registry: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/container_registry